In [ ]:
!pip install -q --upgrade Pillow timm facenet-pytorch kaggle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 140.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

In [ ]:
import json, os

# Fill these in
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": "suyashjaiswal005", "key": "KGAT_c2842e6d8262e2a37714114438b4f390"}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data
!unzip -q /content/data/140k-real-and-fake-faces.zip -d /content/data

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:30<00:00, 132MB/s]



In [ ]:
import os

# Auto-find the train folder
for root, dirs, files in os.walk("/content/data"):
    if "train" in dirs:
        DATA_DIR = root
        print(f"✅ Found DATA_DIR: {DATA_DIR}")
        print(f"   Subfolders: {os.listdir(DATA_DIR)}")
        break

✅ Found DATA_DIR: /content/data/real_vs_fake/real-vs-fake
   Subfolders: ['valid', 'train', 'test']


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm import tqdm

BATCH_SIZE = 32
EPOCHS     = 5
LR         = 1e-4
IMG_SIZE   = 224
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")  # must say cuda

Device: cuda


In [ ]:
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = datasets.ImageFolder(f"{DATA_DIR}/train", transform=transform_train)
val_dataset   = datasets.ImageFolder(f"{DATA_DIR}/valid", transform=transform_val)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Classes:", train_dataset.classes)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Classes: ['fake', 'real']
Train: 100000 | Val: 20000


In [ ]:
model = timm.create_model("efficientnet_b4", pretrained=True, num_classes=2)
model = model.to(DEVICE)
print("Model ready ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Model ready ✅


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
best_val_acc = 0

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")

    model.train()
    total_loss, correct = 0, 0
    for imgs, labels in tqdm(train_loader, desc="Training"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == labels).sum().item()
    print(f"Train Loss: {total_loss/len(train_loader):.4f} | Acc: {correct/len(train_dataset)*100:.2f}%")

    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="Validating"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            total_loss += criterion(out, labels).item()
            correct += (out.argmax(1) == labels).sum().item()
    val_acc = correct / len(val_dataset)
    print(f"Val   Loss: {total_loss/len(val_loader):.4f} | Acc: {val_acc*100:.2f}%")

    scheduler.step()
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/content/drive/MyDrive/deepfake_model.pth")
        print("✅ Saved best model to Drive!")

print(f"\n🏁 Done! Best Val Acc: {best_val_acc*100:.2f}%")

Mounted at /content/drive

--- Epoch 1/5 ---


Training: 100%|██████████| 3125/3125 [23:44<00:00,  2.19it/s]


Train Loss: 0.1855 | Acc: 92.39%


Validating: 100%|██████████| 625/625 [01:25<00:00,  7.32it/s]


Val   Loss: 0.0368 | Acc: 98.69%
✅ Saved best model to Drive!

--- Epoch 2/5 ---


...

Training: 100%|██████████| 3125/3125 [23:47<00:00,  2.19it/s]


Train Loss: 0.0119 | Acc: 99.58%


Validating: 100%|██████████| 625/625 [01:24<00:00,  7.38it/s]


Val   Loss: 0.0079 | Acc: 99.70%
✅ Saved best model to Drive!

--- Epoch 4/5 ---


Training:  88%|████████▊ | 2758/3125 [21:01<02:47,  2.20it/s]

In [2]:
import json, os

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": "suyashjaiswal005", "key": "KGAT_c2842e6d8262e2a37714114438b4f390"}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# This is a preprocessed FF++ dataset with face crops already done
!kaggle datasets download -d greatgamedota/faceforensics-face-images-all -p /content/data
!unzip -q /content/data/faceforensics-face-images-all.zip -d /content/data
!ls /content/data

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
unzip:  cannot find or open /content/data/faceforensics-face-images-all.zip, /content/data/faceforensics-face-images-all.zip.zip or /content/data/faceforensics-face-images-all.zip.ZIP.
ls: cannot access '/content/data': No such file or directory
